In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [46]:
class PositionalEncoding(nn.Module):
    """
    Create a Positional Encoding Layer
    """

    def __init__(self):
        super().__init__()

    # Recieve a 2d matrix
    def forward(self, input_tensor: torch.tensor) -> torch.tensor:
        self.num_of_tokens = input_tensor.shape[1]
        self.d_model = input_tensor.shape[2]
        vector1 = torch.arange(self.num_of_tokens).reshape(-1, 1).float()
        # Vector from 0 to n-1 Shape(n, 1)

        vector2 = torch.arange(self.d_model/2).reshape(-1, 1)
        vector3 = torch.e**(-2*vector2*math.log(10000)/self.d_model).T
        # Vector for division terms Shape(1, d_model)

        matrix = torch.matmul(vector1, vector3)
        # Matrix with shape(num_of_tokens, d_model/2)

        cosine_matrix = torch.cos(matrix)
        sine_matrix = torch.sin(matrix)
        
        # 1. Create an empty matrix of shape (num_of_tokens, d_model)
        pe_matrix = torch.zeros(self.num_of_tokens, self.d_model)

        # 2. Fill the even columns (0, 2, 4...) with sine_matrix
        pe_matrix[:, 0::2] = sine_matrix

        # 3. Fill the odd columns (1, 3, 5...) with cosine_matrix
        pe_matrix[:, 1::2] = cosine_matrix
        
        output_tensor = input_tensor + pe_matrix

        return output_tensor

In [47]:
class MultiHeadAttention(nn.Module):
    """
    Compute Scaled Dot Product Attention using Multiple Heads
    """

    def __init__(self, d_model:int, num_heads: int):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.w_q = nn.Linear(self.d_model, self.d_model, bias=False)
        self.w_k = nn.Linear(self.d_model, self.d_model, bias=False)
        self.w_v = nn.Linear(self.d_model, self.d_model, bias=False)
        self.w_o = nn.Linear(self.d_model, self.d_model, bias=False)

        # Checking if d_model is divisible by num_heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.head_dim = d_model // num_heads # Dimension of query, key and value vectors within each head

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        
        # Checking if input has correct embedding dimension
        if input.shape[-1] != self.d_model:
            raise ValueError(f"Expected input feature dimension to be d_model ({self.d_model}), but got {input.shape[-1]}")

        # Input Dimension: (batch_size, num_tokens, d_model)
        batch_size = input.shape[0]
        num_tokens = input.shape[1]

        # Creating query, key and value vectors from input embeddings. 
        # These vectors contain queries from all the heads combined.
        q = self.w_q(input)
        k = self.w_k(input)
        v = self.w_v(input)
        # Dimension: (batch_sizee, num_tokens, d_model)

        # Splitting our vectors into heads by reshaping our tensors.
        q_head = q.view(batch_size, num_tokens, self.num_heads, self.head_dim)
        k_head = k.view(batch_size, num_tokens, self.num_heads, self.head_dim)
        v_head = v.view(batch_size, num_tokens, self.num_heads, self.head_dim)
        # Dimension: (batch_size, num_tokens, num_heads, head_dimension)

        # Swapping (num_tokens) and (num_heads) dimensions.
        # We do this because we need (num_tokens) and (head_dimension) at the end.
        # This is because we need to perform matrix operations on the last two dimensions
        q_head = q_head.transpose(1, 2)
        k_head = k_head.transpose(1, 2)
        v_head = v_head.transpose(1, 2)
        # Dimension: (batch_size, num_heads, num_tokens, head_dimension)

        # Getting a transpose of key vectors for matrix multiplication 
        k_t = k_head.transpose(-2, -1)
        sim1 = (q_head @ k_t)/math.sqrt(self.head_dim) # Scaled dot product of query and key vectors
        # sim1 is the matrix of attention scores for each token
        # Dimension: (batch_size, num_heads, num_queries, num_keys)

        # Performing softmax along the keys dimension to get attention scores for each key
        sim2 = F.softmax(sim1, dim=-1)
        # Dimension: (batch_size, num_heads, num_queries, num_keys)

        # Weighted sum of attention weights and value vectors
        sim3 = sim2 @ v_head
        # Dimension: (batch_size, num_heads, num_queries, head_dimension)

        # Swapping (num_heads) and (num_queries) dimension back so that we can recombine all the heads
        sim3 = sim3.transpose(1, 2).contiguous()
        sim3 = sim3.view(batch_size, num_tokens, self.d_model)
        # Dimension: (batch_size, num_tokens, d_model)
        # Same as input dimension
        
        # Passing all of our heads into a final layer so that all heads can interact.
        output = self.w_o(sim3)
        # Dimension: (batch_size, num_tokens, d_model)
        # Same as input dimension
        
        return output

In [48]:
class FFNN(nn.Module):
    """
    Apply position-wise feed-forward network.
    """
    def __init__(self, d_model: int, hidden_dim: int, dropout_rate: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.hidden_dim = hidden_dim
        self.hidden_layer = nn.Linear(d_model, hidden_dim)
        self.output_layer = nn.Linear(hidden_dim, d_model)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(p=dropout_rate)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        
        if x.shape[-1] != self.d_model:
            raise ValueError(f"Expected input feature dimension to be d_model ({self.d_model}), but got {x.shape[-1]}")

        z1 = self.hidden_layer(x)
        a1 = self.activation(z1)
        a1 = self.dropout(a1) # Randomly zeroes out some neurons to prevent overfitting
        output = self.output_layer(a1)

        return output

In [ ]:
class EncoderBlock(nn.Module):

    def __init__(self, d_model: int, num_heads: int, hidden_dim: int, dropout_rate: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.hidden_dim = hidden_dim 
        self.attention_layer = MultiHeadAttention(self.d_model, self.num_heads)
        self.ffnn = FFNN(self.d_model, self.hidden_dim, dropout_rate=dropout_rate)
        self.norm1= nn.LayerNorm(self.d_model)
        self.norm2 = nn.LayerNorm(self.d_model)
        self.dropout = nn.Dropout(p=dropout_rate)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: Ouput of Positional Encoding
        # Dimension: (batch_size, num_tokens, d_model)
    
        # ----ATTENTION----
        # Applying Pre-Norm: Normalizing Before Residual Connection
        attn_input = self.norm1(x)
        # Normalizing First

        attn_output = self.attention_layer(attn_input)
        # Passing though attention layer
        
        x = x + self.dropout(attn_output)
        # Applying Residual Connection

        # ----FFNN----
        ffnn_input = self.norm2(x)
        # Normalizing
        ffnn_output = self.ffnn(ffnn_input)
        # Passing through network
        output = x + self.dropout(ffnn_output)
        # Applying Residual Connection
        
        # Dimension remains same: (batch_size, num_tokens, d_model)
        return output


In [50]:
batch_size = 2
num_tokens = 3
d_model = 8
num_heads = 2
head_dim = 4
hidden_dim = 16

In [51]:
dummy_block_input = torch.rand(batch_size, num_tokens, d_model)
print(dummy_block_input.shape)
print("(batch_size, num_tokens, embedding_dimension)")
dummy_block_input

torch.Size([2, 3, 8])
(batch_size, num_tokens, embedding_dimension)


tensor([[[0.1934, 0.6178, 0.8076, 0.0451, 0.2919, 0.9304, 0.4824, 0.7981],
         [0.9079, 0.8815, 0.6050, 0.4348, 0.6578, 0.3595, 0.7211, 0.9252],
         [0.4690, 0.4751, 0.4657, 0.3984, 0.7559, 0.4472, 0.6074, 0.3340]],

        [[0.6492, 0.3176, 0.3747, 0.4053, 0.1262, 0.9062, 0.9621, 0.3411],
         [0.0254, 0.6090, 0.3339, 0.2332, 0.8789, 0.4452, 0.2143, 0.6491],
         [0.0197, 0.5012, 0.5195, 0.0615, 0.9836, 0.8530, 0.9176, 0.5313]]])

In [52]:
encoder = EncoderBlock(d_model, num_heads, hidden_dim)
encoder

EncoderBlock(
  (attention_layer): MultiHeadAttention(
    (w_q): Linear(in_features=8, out_features=8, bias=False)
    (w_k): Linear(in_features=8, out_features=8, bias=False)
    (w_v): Linear(in_features=8, out_features=8, bias=False)
    (w_o): Linear(in_features=8, out_features=8, bias=False)
  )
  (ffnn): FFNN(
    (hidden_layer): Linear(in_features=8, out_features=16, bias=True)
    (output_layer): Linear(in_features=16, out_features=8, bias=True)
    (activation): ReLU()
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (norm1): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
  (norm2): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
  (dropout): Dropout(p=0.1, inplace=False)
)

In [53]:
dummy_output = encoder(dummy_block_input)
dummy_output

tensor([[[ 0.0529,  0.5422,  0.9296,  0.3462,  0.0607,  0.7432,  0.0480,
           0.7440],
         [ 0.6294,  0.5714,  0.5276,  0.1239,  0.3818, -0.0557,  0.4178,
           1.0569],
         [ 0.2056,  0.1431,  0.7194,  0.5875,  0.4671, -0.0405,  0.5536,
           0.2632]],

        [[ 0.4736,  0.3808,  0.7884, -0.0930,  0.4921,  0.9062,  1.0319,
          -0.2208],
         [-0.0677,  0.4358,  0.9147,  0.5524,  0.7057,  0.1148,  0.2143,
           0.8269],
         [-0.2010,  0.3911,  0.7185, -0.2628,  1.0027,  0.4305,  0.8932,
          -0.0827]]], grad_fn=<AddBackward0>)

In [ ]:
# Now, Stack Multiple Encoders:
class EncoderStack(nn.Module):
    
    def __init__(self, d_model: int, num_heads: int, hidden_dim: int, num_blocks: int, dropout_rate: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.layers = nn.ModuleList([EncoderBlock(d_model, num_heads, hidden_dim) for _ in range(num_blocks)])
        self.norm_1 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(p=dropout_rate)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Dimension of tokens: (batch_size, num_tokens) 

        for layer in self.layers:
            x = layer(x)
        output = self.norm_1(x)
        # Dimension: (Batch_size, num_tokens, d_model)

        return output

In [ ]:
batch_size = 2
num_tokens = 3
d_model = 8
num_heads = 2
head_dim = 4
vocab_size = 5
hidden_dim = 16
num_blocks = 6

In [56]:
dummy_stack_input = torch.randint(low=0, high=vocab_size, size=(batch_size, num_tokens))
print(dummy_stack_input.shape)
print("(batch_size, num_tokens, embedding_dimension)")
dummy_stack_input

torch.Size([2, 3])
(batch_size, num_tokens, embedding_dimension)


tensor([[3, 2, 0],
        [1, 1, 3]])

In [ ]:
encoder_stack = EncoderStack(vocab_size, d_model, num_heads, hidden_dim, num_blocks)
encoder_stack

EncoderStack(
  (embedding): Embedding(5, 8)
  (positional_encoding): PositionalEncoding()
  (layers): ModuleList(
    (0-5): 6 x EncoderBlock(
      (attention_layer): MultiHeadAttention(
        (w_q): Linear(in_features=8, out_features=8, bias=False)
        (w_k): Linear(in_features=8, out_features=8, bias=False)
        (w_v): Linear(in_features=8, out_features=8, bias=False)
        (w_o): Linear(in_features=8, out_features=8, bias=False)
      )
      (ffnn): FFNN(
        (hidden_layer): Linear(in_features=8, out_features=16, bias=True)
        (output_layer): Linear(in_features=16, out_features=8, bias=True)
        (activation): ReLU()
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (norm1): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (norm_1): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
  (dropout): Dropout(p=0.1, inplac

In [58]:
encoder_stack_output = encoder_stack(dummy_stack_input)
print(encoder_stack_output.shape)
encoder_stack_output

torch.Size([2, 3, 8])


tensor([[[ 0.3526,  0.4894, -0.0849, -0.2255, -1.7720, -0.3188, -0.4696,
           2.0288],
         [ 0.4799,  1.1191,  0.9573,  0.0776,  0.7290, -1.8780, -1.2084,
          -0.2766],
         [ 0.4327, -0.4302,  0.9077,  0.8385,  0.3286,  1.0423, -1.7030,
          -1.4164]],

        [[ 0.8470,  1.1957, -0.6840,  1.3008, -0.5434, -1.7246,  0.2206,
          -0.6121],
         [-0.2688,  1.5486, -0.6836,  1.5827, -0.7449, -1.3837,  0.1831,
          -0.2335],
         [ 0.5073,  0.4836,  0.1733, -1.6815, -0.9877,  0.0441, -0.4112,
           1.8721]]], grad_fn=<NativeLayerNormBackward0>)